# MYRIAD production run — results analysis v1

Summary plots for `production/myriad_conus_20240101_20251231` (CONUS 50 km grid, hourly, 2024-01-01 to 2025-12-31).

1. Per-hazard occurrence density over CONUS
2. Total co-located hazard-pair counts (log scale) — pair hotspots
3. Dominant pair among the five most frequent ordered pairs, per cell
4. Ordered hazard-pair matrix (hazard that starts first on the vertical axis)

Spatial footprints come from the `checkpoints/labels_*.npz` label cubes, so pair hotspots reflect actual
spatial overlap of the two events rather than centroid proximity.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.lines import Line2D
from scipy import sparse

DATA = Path(
    "/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/"
    "NASA ROSES Disasters 2025-2027/data"
)
RUN = DATA / "MYRIAD_events_output/production/myriad_conus_20240101_20251231"
CKPT = RUN / "checkpoints"

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = REPO / "notebooks/outputs/myriad_results_v1"
OUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 9})
print(RUN.exists(), CKPT.exists(), OUT)

In [ ]:
# Analysis grid (EPSG:5070, 50 km) and CONUS state boundaries for context.
grid = xr.open_dataset(REPO / "data/analysis_grid_CONUS_50km.nc")
NY, NX = grid["lat"].shape
NCELL = NY * NX
gx, gy = grid["x"].values, grid["y"].values
dx, dy = gx[1] - gx[0], gy[1] - gy[0]
XE = np.append(gx - dx / 2, gx[-1] + dx / 2)
YE = np.append(gy - dy / 2, gy[-1] + dy / 2)

_states_cache = REPO / "data/cb_2023_us_state_20m.geojson"
try:
    if _states_cache.exists():
        STATES = gpd.read_file(_states_cache)
    else:
        STATES = gpd.read_file(
            "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_state_20m.zip"
        )
        STATES.to_file(_states_cache, driver="GeoJSON")
    STATES = STATES[~STATES["STUSPS"].isin(["AK", "HI", "PR"])].to_crs("EPSG:5070")
except Exception as exc:  # basemap is decorative; plots still work without it
    print(f"State boundaries unavailable ({exc}); plotting without basemap.")
    STATES = None


def frame(ax, title=""):
    if STATES is not None:
        STATES.boundary.plot(ax=ax, color="0.3", linewidth=0.35, zorder=3)
    ax.set(xlim=(XE[0], XE[-1]), ylim=(YE[0], YE[-1]), xticks=[], yticks=[], title=title)
    ax.set_aspect("equal")
    for spine in ax.spines.values():
        spine.set_visible(False)


print(f"grid {NY} x {NX} = {NCELL} cells")

In [ ]:
events = pd.read_csv(
    RUN / "single_hazard_events_all.csv", parse_dates=["start_time", "end_time"]
)
events["row"] = np.arange(len(events))
ROW_OF = dict(zip(events["event_id"], events["row"]))
HAZARDS = sorted(events["hazard"].unique())
LABELS = {h: h.replace("_", " ") for h in HAZARDS}

print(events["hazard"].value_counts().to_string())

## 1. Per-hazard occurrence density

One pass per hazard over its label cube produces two per-cell fields:

- **hazard-hours** — number of hourly time steps the cell was inside any event of that hazard
- **event count** — number of distinct events that touched the cell (counted as label transitions)

The same pass builds a sparse `event x cell` footprint matrix used for the pair analysis below.

In [ ]:
hazard_hours, hazard_counts = {}, {}
_rows, _cols = [], []

for hz in HAZARDS:
    path = CKPT / f"labels_{hz}.npz"
    if not path.exists():
        print(f"skip {hz}: no label checkpoint")
        continue
    lab = np.load(path)["labels"]

    hazard_hours[hz] = (lab > 0).sum(axis=0).astype(np.int32)

    starts = np.empty(lab.shape, dtype=bool)
    starts[0] = lab[0] > 0
    starts[1:] = (lab[1:] > 0) & (lab[1:] != lab[:-1])
    hazard_counts[hz] = starts.sum(axis=0).astype(np.int32)
    del starts

    tt, yy, xx = np.nonzero(lab)
    lab_id = lab[tt, yy, xx].astype(np.int64)
    cell = yy.astype(np.int64) * NX + xx
    uniq = np.unique(lab_id * NCELL + cell)  # one entry per (event, cell)
    ev_num, ev_cell = uniq // NCELL, uniq % NCELL
    del lab, tt, yy, xx, lab_id, cell, uniq

    sub = events.loc[events["hazard"] == hz]
    nums = sub["event_id"].str.rsplit("_", n=1).str[-1].astype(int).values
    lut = np.full(int(max(nums.max(), ev_num.max())) + 1, -1, dtype=np.int64)
    lut[nums] = sub["row"].values
    rid = lut[ev_num]
    ok = rid >= 0
    _rows.append(rid[ok])
    _cols.append(ev_cell[ok])
    print(f"{hz:24s} events={len(sub):5d}  footprint entries={ok.sum():,}")

rows = np.concatenate(_rows)
cols = np.concatenate(_cols)
FOOTPRINT = sparse.csr_matrix(
    (np.ones(rows.size, np.float32), (rows, cols)), shape=(len(events), NCELL)
)
FOOTPRINT.data[:] = 1.0
del _rows, _cols, rows, cols
print(f"footprint matrix {FOOTPRINT.shape}, nnz={FOOTPRINT.nnz:,}")

In [ ]:
field, unit = hazard_hours, "event-hours per cell"  # swap to hazard_counts for event counts
keys = [h for h in HAZARDS if h in field]
ncol = 3
nrow = int(np.ceil(len(keys) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 2.9 * nrow))

for ax, hz in zip(axes.ravel(), keys):
    arr = field[hz].astype(float)
    arr[arr == 0] = np.nan
    # per-panel percentile stretch: most hazards cover nearly every cell, so a
    # fixed vmin=1 saturates the whole map at the top of the colour scale
    vmin = max(np.nanpercentile(arr, 2), 1)
    vmax = np.nanpercentile(arr, 99.5)
    if vmax <= vmin:
        vmin, vmax = 1, max(np.nanmax(arr), 2)
    mesh = ax.pcolormesh(
        XE, YE, arr, cmap="YlOrRd",
        norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), shading="flat", zorder=1,
    )
    frame(ax, f"{LABELS[hz]}  (n={int((events['hazard'] == hz).sum()):,})")
    fig.colorbar(mesh, ax=ax, fraction=0.035, pad=0.02, extend="both")

for ax in axes.ravel()[len(keys):]:
    ax.axis("off")

fig.suptitle(f"MYRIAD single-hazard occurrence density, 2024-2025 ({unit}, log scale)", y=0.995)
fig.tight_layout()
fig.savefig(OUT / "hazard_density_panel.png", bbox_inches="tight")
plt.show()

## 2. Hazard-pair hotspots

Each row of `multi_hazard_event_links.csv` is a linked pair of single-hazard events. Pairs are ordered by
start time, so `hazard_1` is the hazard that begins first. A pair is counted in a grid cell when the
footprints of both member events include that cell.

In [ ]:
links = pd.read_csv(RUN / "multi_hazard_event_links.csv")
start = events.set_index("event_id")["start_time"]
ta = links["event_id_a"].map(start).values
tb = links["event_id_b"].map(start).values
first_is_b = tb < ta

links["event_1"] = np.where(first_is_b, links["event_id_b"], links["event_id_a"])
links["event_2"] = np.where(first_is_b, links["event_id_a"], links["event_id_b"])
links["hazard_1"] = np.where(first_is_b, links["hazard_b"], links["hazard_a"])
links["hazard_2"] = np.where(first_is_b, links["hazard_a"], links["hazard_b"])
links["row_1"] = links["event_1"].map(ROW_OF)
links["row_2"] = links["event_2"].map(ROW_OF)
links = links.dropna(subset=["row_1", "row_2"]).astype({"row_1": int, "row_2": int})

pair_counts = links.groupby(["hazard_1", "hazard_2"]).size().sort_values(ascending=False)
print(f"{len(links):,} ordered pairs across {pair_counts.size} hazard-pair types")
print(pair_counts.head(10).to_string())

In [ ]:
def pair_map(sub):
    """Per-cell count of pairs whose two member footprints both cover the cell."""
    total = np.zeros(NCELL, dtype=np.float64)
    for lo in range(0, len(sub), 20000):  # chunked to cap peak memory
        blk = sub.iloc[lo : lo + 20000]
        overlap = FOOTPRINT[blk["row_1"].values].multiply(FOOTPRINT[blk["row_2"].values])
        total += np.asarray(overlap.sum(axis=0)).ravel()
    return total.reshape(NY, NX)


pair_maps = {p: pair_map(g) for p, g in links.groupby(["hazard_1", "hazard_2"])}
total_pairs = np.sum(list(pair_maps.values()), axis=0)
print(f"max cell pair count = {total_pairs.max():,.0f}")

In [ ]:
arr = total_pairs.astype(float)
arr[arr == 0] = np.nan

fig, ax = plt.subplots(figsize=(9, 5.6))
mesh = ax.pcolormesh(
    XE, YE, arr, cmap="YlOrRd",
    norm=mcolors.LogNorm(vmin=1, vmax=np.nanmax(arr)), shading="flat", zorder=1,
)
frame(ax, "Co-located multi-hazard pairs per 50 km cell, 2024-2025")
cbar = fig.colorbar(mesh, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("number of hazard pairs (log scale)")
fig.tight_layout()
fig.savefig(OUT / "pair_hotspots_total_log.png", bbox_inches="tight")
plt.show()

### Dominant pair among the five most frequent combinations

Each cell is coloured by whichever of the top-five ordered pairs occurs most often there. Cells where none
of the five occurs are left blank. The panels below show each pair's own count field for context.

In [ ]:
top5 = list(pair_counts.head(5).index)
stack = np.stack([pair_maps[p] for p in top5])
totals = stack.sum(axis=0)
winner = stack.argmax(axis=0).astype(float)
winner[totals == 0] = np.nan

colors = plt.get_cmap("tab10").colors[:5]
cmap = mcolors.ListedColormap(colors)

fig, ax = plt.subplots(figsize=(9, 6.4))
ax.pcolormesh(XE, YE, winner, cmap=cmap, vmin=-0.5, vmax=4.5, shading="flat", zorder=1)
frame(ax, "Most frequent hazard pair per cell (top-5 pairs), 2024-2025")
ax.legend(
    handles=[
        Line2D([], [], marker="s", ls="", markersize=9, color=colors[i],
               label=f"{LABELS[a]} \u2192 {LABELS[b]}  ({pair_counts[(a, b)]:,})")
        for i, (a, b) in enumerate(top5)
    ],
    loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=False,
    fontsize=8, title="first hazard \u2192 second hazard", title_fontsize=8,
)
fig.savefig(OUT / "pair_dominant_top5.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, (a, b) in zip(axes.ravel(), top5):
    f = pair_maps[(a, b)].astype(float)
    f[f == 0] = np.nan
    mesh = ax.pcolormesh(
        XE, YE, f, cmap="YlOrRd",
        norm=mcolors.LogNorm(vmin=1, vmax=np.nanmax(f)), shading="flat", zorder=1,
    )
    frame(ax, f"{LABELS[a]} \u2192 {LABELS[b]}")
    fig.colorbar(mesh, ax=ax, fraction=0.035, pad=0.02)
axes.ravel()[-1].axis("off")
fig.suptitle("Top-5 hazard pairs, per-cell counts (log scale)", y=0.99)
fig.tight_layout()
fig.savefig(OUT / "pair_top5_panels.png", bbox_inches="tight")
plt.show()

## 3. Ordered hazard-pair matrix

Rows are the hazard that starts first, columns the hazard that follows. The matrix is asymmetric by
construction; the diagonal counts pairs of two distinct events of the same hazard type.

In [ ]:
matrix = (
    links.groupby(["hazard_1", "hazard_2"]).size().unstack(fill_value=0)
    .reindex(index=HAZARDS, columns=HAZARDS, fill_value=0)
)

vals = matrix.values.astype(float)
shown = np.where(vals > 0, vals, np.nan)

fig, ax = plt.subplots(figsize=(7.6, 6.2))
mesh = ax.imshow(
    shown, cmap="YlGnBu",
    norm=mcolors.LogNorm(vmin=1, vmax=np.nanmax(shown)), origin="upper",
)
ax.set_xticks(range(len(HAZARDS)))
ax.set_xticklabels([LABELS[h] for h in HAZARDS], rotation=40, ha="right")
ax.set_yticks(range(len(HAZARDS)))
ax.set_yticklabels([LABELS[h] for h in HAZARDS])
ax.set_xlabel("second hazard")
ax.set_ylabel("first hazard (starts earlier)")
ax.set_title("Ordered hazard-pair co-occurrence counts, 2024-2025")

for i in range(len(HAZARDS)):
    for j in range(len(HAZARDS)):
        v = vals[i, j]
        if v > 0:
            ax.text(
                j, i, f"{int(v):,}", ha="center", va="center", fontsize=7,
                color="white" if v > np.nanmax(shown) * 0.25 else "0.15",
            )

fig.colorbar(mesh, ax=ax, fraction=0.04, pad=0.02, label="number of pairs (log scale)")
fig.tight_layout()
fig.savefig(OUT / "pair_matrix_ordered.png", bbox_inches="tight")
plt.show()

matrix.to_csv(OUT / "pair_matrix_ordered.csv")
matrix